# FACE-RLVR Patient Vignette Explorer

Interactive notebook for extracting structured clinical data and rendering
full French vignettes for any of the four FACE cohorts:

- **BP** — Bipolar Disorder
- **SZ** — Schizophrenia
- **DR** — Treatment-Resistant Depression
- **ASP** — Autism Spectrum Disorder (TSASDI)

## Architecture (YAML-driven)

All clinical knowledge — instrument definitions, severity thresholds, lab
reference ranges, categorical code lookups, clinical computation constants
(BMI, metabolic syndrome, Framingham, drug interactions, cognitive norms) **and**
every CSV column name the extractors read — lives in YAML files under
`config/glossary/`, validated at load time by Pydantic v2 models.

The Python extractors contain only **logic**: how to decode sex, how to compute
derived fields, how to assemble dataclasses. They never hard-code a column name.

```
config/glossary/
├── common/
│   ├── instruments.yaml         (16 shared instruments)
│   ├── thresholds.yaml          (8 reusable severity band lists)
│   ├── clinical_constants.yaml  (BMI, metsyn, Framingham, drug rules, ...)
│   └── categorical_codes.yaml   (MARITAL, EDUCATION, EMPLOYMENT)
├── bp/ sz/ dr/ asp/
│   ├── instruments.yaml         (cohort-specific + overrides)
│   ├── lab_ranges.yaml          (lab reference ranges)
│   ├── column_map.yaml          (every CSV column the extractor reads)
│   └── (dr, asp only) categorical_codes.yaml
```

## What this notebook demonstrates

1. Shared helpers to load any cohort and render its vignette
2. Cohort-by-cohort walkthrough showing the structured dataclass
3. Cohort-specific fields unique to each pathology
4. Inspecting the loaded YAML glossary directly
5. Multi-patient comparison as a pandas DataFrame
6. Live YAML mutation demo proving the data-driven design
7. Exporting a vignette to markdown for clinical review

## 0. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Ensure the project source is on the path when running locally
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_DIR = REPO_ROOT / "data"
print(f"Repo root:  {REPO_ROOT}")
print(f"Data dir:   {DATA_DIR}")
print(f"CSVs:       {sorted(p.name for p in DATA_DIR.glob('*.csv'))}")

Repo root:  /Users/andriikulakovskyi/Desktop/llm-rl/psych-dataset
Data dir:   /Users/andriikulakovskyi/Desktop/llm-rl/psych-dataset/data
CSVs:       ['ASP.csv', 'BP.csv', 'DR.csv', 'SZ.csv']


In [2]:
from face_rlvr.profiles import (
    extract_bp_patient, build_bp_profile,
    extract_sz_patient, build_sz_profile,
    extract_dr_patient, build_dr_profile,
    extract_asp_patient, build_asp_profile,
)

COHORTS = {
    "BP":  {"csv": DATA_DIR / "BP.csv",  "extract": extract_bp_patient,  "build": build_bp_profile,  "label": "Bipolar Disorder"},
    "SZ":  {"csv": DATA_DIR / "SZ.csv",  "extract": extract_sz_patient,  "build": build_sz_profile,  "label": "Schizophrenia"},
    "DR":  {"csv": DATA_DIR / "DR.csv",  "extract": extract_dr_patient,  "build": build_dr_profile,  "label": "Treatment-Resistant Depression"},
    "ASP": {"csv": DATA_DIR / "ASP.csv", "extract": extract_asp_patient, "build": build_asp_profile, "label": "Autism Spectrum Disorder (TSASDI)"},
}

def load_cohort(name: str, nrows: int | None = None) -> pd.DataFrame:
    """Load a cohort CSV into a DataFrame. Use nrows for quick exploration."""
    return pd.read_csv(COHORTS[name]["csv"], nrows=nrows, low_memory=False)

def build_vignette(cohort: str, row: pd.Series):
    """Extract patient data + build full vignette for one CSV row."""
    cfg = COHORTS[cohort]
    data = cfg["extract"](row)
    profile = cfg["build"](data)
    return profile, data

def print_header(title: str) -> None:
    print("=" * 80)
    print(f"  {title}")
    print("=" * 80)

for name, cfg in COHORTS.items():
    print(f"  {name:3s}  {cfg['label']:38s}  {cfg['csv'].name}")

  BP   Bipolar Disorder                        BP.csv
  SZ   Schizophrenia                           SZ.csv
  DR   Treatment-Resistant Depression          DR.csv
  ASP  Autism Spectrum Disorder (TSASDI)       ASP.csv


## 1. Quick end-to-end check

Render the vignette for the first patient in each cohort. Confirms that the
full pipeline (YAML → extractor → profile builder) works on real data.

In [3]:
for cohort in COHORTS:
    df = load_cohort(cohort, nrows=1)
    profile, data = build_vignette(cohort, df.iloc[0])
    print_header(f"{cohort} — patient_id={data.patient_id}  ({len(profile.full_vignette)} chars)")
    # Print just the synthesis + demographics for brevity
    print(profile.synthesis_section)
    print()
    print(profile.demographics_section)
    print()

  BP — patient_id=10001  (4754 chars)
Synthèse clinique : Patient suivie pour Bipolaire de type 2, actuellement euthymique (MADRS = 2). Facteurs notables : impulsivité élevée, troubles du sommeil.

Patient 10001, femme de 22 ans, suivie pour Bipolaire de type 2.
Célibataire, niveau d'études : bac.

  SZ — patient_id=10001  (2694 chars)
Synthèse clinique : Patient suivie pour Schizophrénie, symptômes psychotiques — symptômes légers (PANSS = 45).

Patient 10001, femme de 38 ans, suivie pour Schizophrénie.
Niveau d'études : bac+2.

  DR — patient_id=10035  (4863 chars)
Synthese clinique : Patient suivie pour depression resistante, actuellement en depression moderee (MADRS = 29). Niveau de resistance : resistance partielle. Risque suicidaire : ideation avec methode (C-SSRS niveau 3). Facteurs notables : troubles du sommeil. Traitement : ECT.

Patient 10035, femme de 45 ans, suivie pour Trouble dépressif majeur.
En couple, niveau d'etudes : bep, employé(e).

  ASP — patient_id=10004  (2471 

## 2. Bipolar Disorder (BP)

BP is the largest cohort (~5,400 patients, 2,229 columns). Key features:

- **Mood scores**: MADRS (depression, hetero-rated), YMRS (mania), CGI-S,
  ASRM (self-report hypomania), MAThyS (dimensional), QIDS-SR16
- **Episode history**: lifetime counts of depressive, manic, hypomanic, mixed,
  and psychotic episodes; rapid cycling; DSM-5 current episode criteria
- **Mood-stabilizer monitoring**: lithium/valproate/carbamazepine plasma levels
- **V1 follow-up**: only BP has `_n1` follow-up columns for MADRS and YMRS
- **BP-specific**: DIVA ADHD structured interview, circadian rhythm
- **Impulsivity + hostility**: BIS-10, BDHI, ALS, AIM

In [4]:
df_bp = load_cohort("BP", nrows=1)
profile_bp, data_bp = build_vignette("BP", df_bp.iloc[0])

print_header("BP: Mood scores (hetero + self-report)")
for key, interp in data_bp.mood_scores.items():
    if interp.score_available:
        marker = " ⚠" if interp.suspect_value else ""
        print(f"  {key:12s}  {interp.raw_score!s:6s}  [{interp.severity_code}]{marker}")

print_header("BP: Current episode DSM criteria (etatd/etatm)")
ce = data_bp.current_episode_criteria
print(f"  Depressive symptoms: {ce.depressive_symptom_count}/9")
print(f"    depressed_mood={ce.depressed_mood}, anhedonia={ce.anhedonia}, sleep_disturbance={ce.sleep_disturbance}")
print(f"  Manic symptoms:      {ce.manic_symptom_count}/9")
print(f"    elevated_mood={ce.elevated_mood}, grandiosity={ce.grandiosity}, flight_of_ideas={ce.flight_of_ideas}")

print_header("BP: Psychiatric history + rapid cycling")
ph = data_bp.psychiatric_history
print(f"  First episode:       {ph.age_first_episode} y/o")
print(f"  Rapid cycling:       {ph.rapid_cycling}")
print(f"  Lifetime episodes:   depressive={ph.n_depressive_episodes_lifetime}, "
      f"manic={ph.n_manic_episodes_lifetime}, hypomanic={ph.n_hypomanic_episodes_lifetime}")

print_header("BP: Treatment — mood stabilizer plasma levels")
t = data_bp.treatments
print(f"  Lithium:      on={t.on_lithium},       plasma={t.lithium_plasma}")
print(f"  Valproate:    on={t.on_valproate},     plasma={t.valproate_plasma}")
print(f"  Carbamazepine: on={t.on_carbamazepine}")
print(f"  Lamotrigine:  on={t.on_lamotrigine}")
print(f"  Adherence (MARS): {t.medication_adherence.severity_label_fr if t.medication_adherence and t.medication_adherence.score_available else 'n/a'}")

print_header("BP: V1 follow-up (only BP has _n1 columns)")
if data_bp.v1_mood_scores:
    for key, interp in data_bp.v1_mood_scores.items():
        print(f"  {key}: {interp.raw_score} ({interp.severity_code})")
else:
    print("  No V1 follow-up data for this patient")

print_header("BP: DIVA ADHD structured interview (BP-only)")
diva = data_bp.diva_adhd
print(f"  Adult inattention: {diva.attention_adult_count}/9")
print(f"  Adult hyperactivity: {diva.hyperactivity_adult_count}/9")
print(f"  Childhood inattention: {diva.attention_childhood_count}/9")
print(f"  Childhood hyperactivity: {diva.hyperactivity_childhood_count}/9")

  BP: Mood scores (hetero + self-report)
  MADRS         2.0     [normal]
  YMRS          2.0     [normal]
  CGI-S         3.0     [mild]
  ASRM          7.0     [positive]
  MAThyS        136.1   [activation]
  QIDS-SR16     8.0     [mild]
  BP: Current episode DSM criteria (etatd/etatm)
  Depressive symptoms: 1/9
    depressed_mood=False, anhedonia=False, sleep_disturbance=False
  Manic symptoms:      0/9
    elevated_mood=False, grandiosity=False, flight_of_ideas=False
  BP: Psychiatric history + rapid cycling
  First episode:       16 y/o
  Rapid cycling:       True
  Lifetime episodes:   depressive=None, manic=0, hypomanic=None
  BP: Treatment — mood stabilizer plasma levels
  Lithium:      on=False,       plasma=None
  Valproate:    on=False,     plasma=None
  Carbamazepine: on=False
  Lamotrigine:  on=False
  Adherence (MARS): Observance partielle
  BP: V1 follow-up (only BP has _n1 columns)
  No V1 follow-up data for this patient
  BP: DIVA ADHD structured interview (BP-only)
 

## 3. Schizophrenia (SZ)

SZ uses different instruments + phenomenology. Key features:

- **Psychosis severity**: PANSS total + positive + negative + general (4 subscales)
- **SZ-specific depression scale**: Calgary Depression Scale for Schizophrenia
- **Phenomenology**: 19 boolean psychotic symptom flags (hallucinations,
  delusions, disorganization, negative symptoms, catatonia)
- **Insight assessment**: SUMD (Scale to assess Unawareness of Mental Disorder,
  9 items)
- **Movement disorders**: AIMS (tardive dyskinesia), BARS (akathisia)
- **Clozapine monitoring**: plasma level + on/off flag

In [5]:
df_sz = load_cohort("SZ", nrows=1)
profile_sz, data_sz = build_vignette("SZ", df_sz.iloc[0])

print_header("SZ: PANSS (4 scores) + Calgary depression")
for key in ("PANSS", "PANSS-P", "PANSS-N", "PANSS-G"):
    interp = data_sz.psychosis_scores.get(key)
    if interp and interp.score_available:
        print(f"  {key:10s}  {interp.raw_score!s:6s}  [{interp.severity_code}]")
calgary = data_sz.depression_scores.get("Calgary")
if calgary and calgary.score_available:
    print(f"  Calgary     {calgary.raw_score:<6}  [{calgary.severity_code}]")

print_header("SZ: Current psychotic symptoms (phenomenology flags)")
sx = data_sz.psychotic_symptoms
positive = [name for name, val in [
    ("hallucinations_auditory", sx.hallucinations_auditory),
    ("hallucinations_visual", sx.hallucinations_visual),
    ("delusions_persecution", sx.delusions_persecution),
    ("delusions_reference", sx.delusions_reference),
    ("delusions_grandiosity", sx.delusions_grandiosity),
    ("thought_control", sx.thought_control),
    ("thought_broadcasting", sx.thought_broadcasting),
] if val]
negative = [name for name, val in [
    ("avolition", sx.avolition),
    ("alogia", sx.alogia),
] if val]
disorg = [name for name, val in [
    ("disorganized_speech", sx.disorganized_speech),
    ("disorganized_behavior", sx.disorganized_behavior),
    ("bizarre_behavior", sx.bizarre_behavior),
] if val]
print(f"  Positive:      {positive or '—'}")
print(f"  Negative:      {negative or '—'}")
print(f"  Disorganization: {disorg or '—'}")
print(f"  Catatonia:     {sx.catatonia}")

print_header("SZ: Insight (SUMD — 9 items, higher = less insight)")
ins = data_sz.insight
for attr in ("awareness_of_illness", "awareness_of_medication_effect",
             "awareness_of_social_consequences", "awareness_of_hallucinations",
             "awareness_of_delusions", "awareness_of_flat_affect",
             "awareness_of_anhedonia", "awareness_of_asociality"):
    val = getattr(ins, attr, None)
    if val is not None:
        print(f"  {attr:40s}  {val}")

print_header("SZ: Movement disorders (AIMS + BARS)")
for key, label in (("AIMS", "tardive dyskinesia"), ("BARS", "akathisia")):
    interp = data_sz.movement_scores.get(key)
    if interp and interp.score_available:
        print(f"  {key} ({label}): {interp.raw_score}  [{interp.severity_code}]")

print_header("SZ: Clozapine monitoring")
t = data_sz.treatments
print(f"  On clozapine: {t.on_clozapine}")
if t.on_clozapine:
    print(f"  Plasma level: {t.clozapine_plasma}")
print(f"  # antipsychotics:  {t.n_antipsychotics}")
print(f"  # anticholinergics: {t.n_anticholinergics}")
print(f"  # mood stabilizers: {t.n_mood_stabilizers}")

  SZ: PANSS (4 scores) + Calgary depression
  PANSS       45.0    [mild]
  PANSS-P     15.0    [mild]
  PANSS-N     7.0     [absent]
  PANSS-G     23.0    [minimal]
  Calgary     1.0     [none]
  SZ: Current psychotic symptoms (phenomenology flags)
  Positive:      —
  Negative:      —
  Disorganization: —
  Catatonia:     False
  SZ: Insight (SUMD — 9 items, higher = less insight)
  awareness_of_illness                      1.0
  awareness_of_medication_effect            2.0
  awareness_of_social_consequences          1.0
  SZ: Movement disorders (AIMS + BARS)
  AIMS (tardive dyskinesia): 0.0  [none]
  BARS (akathisia): 0.0  [absent]
  SZ: Clozapine monitoring
  On clozapine: False
  # antipsychotics:  2
  # anticholinergics: 0
  # mood stabilizers: 0


## 4. Treatment-Resistant Depression (DR)

DR is the smallest but clinically richest cohort. Key features:

- **Depression assessment**: MADRS + QIDS + ERD (psychomotor retardation) +
  SHAPS (anhedonia)
- **Treatment resistance staging**: `epi_resist` level (0/1/2), Sachs score,
  current episode number, current episode duration, antidepressant response
- **C-SSRS binary ideation**: 5 binary items (wish to die → ideation with
  plan) with derived `highest_ideation_level`
- **Impulsivity**: BIS-10 (newly extracted — data existed but was not read
  pre-refactor)
- **Lithium/valproate levels**, **ECT history**

In [6]:
df_dr = load_cohort("DR", nrows=1)
profile_dr, data_dr = build_vignette("DR", df_dr.iloc[0])

print_header("DR: Depression assessment (4 instruments)")
for key in ("MADRS", "QIDS", "ERD", "SHAPS"):
    interp = data_dr.depression_scores.get(key)
    if interp and interp.score_available:
        marker = " ⚠" if interp.suspect_value else ""
        print(f"  {key:8s}  {interp.raw_score!s:6s}  [{interp.severity_code}]{marker}")

print_header("DR: Treatment resistance staging")
tr = data_dr.treatment_resistance
print(f"  Resistance level:        {tr.resistance_level} (is_resistant={tr.is_resistant})")
print(f"  Current episode number:  {tr.current_episode_number}")
print(f"  Current episode duration (months): {tr.current_episode_duration_months}")
print(f"  Age at first treatment:  {tr.age_first_treatment}")
print(f"  Total treatment duration (months): {tr.total_treatment_duration_months}")
print(f"  Psychotic features:      {tr.has_psychotic_features}")
print(f"  Achieved remission:      {tr.achieved_complete_remission}")
print(f"  Sachs score:             {tr.sachs_score}")

print_header("DR: C-SSRS binary ideation items (highest level wins)")
cssrs = data_dr.cssrs_assessment
print(f"  1. Wish to die:              {cssrs.wish_to_die}")
print(f"  2. Nonspecific ideation:     {cssrs.nonspecific_ideation}")
print(f"  3. Ideation with method:     {cssrs.ideation_with_method}")
print(f"  4. Ideation with intent:     {cssrs.ideation_with_intent}")
print(f"  5. Ideation with plan:       {cssrs.ideation_with_plan}")
print(f"  → Derived highest level:     {cssrs.highest_ideation_level}/5")

print_header("DR: BIS-10 impulsivity (newly extracted)")
bis = data_dr.impulsivity_scores.get("BIS-10")
if bis and bis.score_available:
    print(f"  BIS-10 total: {bis.raw_score} ({bis.severity_code})")
else:
    print("  BIS-10: not available for this patient")

print_header("DR: Treatment + episode counts")
t = data_dr.treatments
print(f"  Lithium level:  {t.lithium_level}")
print(f"  Valproate level: {t.valproate_level}")
print(f"  ECT history:    {t.has_ect}")
print(f"  Lifetime episodes: {data_dr.episode_counts}")

  DR: Depression assessment (4 instruments)
  MADRS     29.0    [moderate]
  QIDS      24.0    [very_severe]
  DR: Treatment resistance staging
  Resistance level:        resistance partielle (is_resistant=True)
  Current episode number:  2
  Current episode duration (months): 27.0
  Age at first treatment:  20
  Total treatment duration (months): 27.0
  Psychotic features:      False
  Achieved remission:      False
  Sachs score:             None
  DR: C-SSRS binary ideation items (highest level wins)
  1. Wish to die:              True
  2. Nonspecific ideation:     True
  3. Ideation with method:     True
  4. Ideation with intent:     False
  5. Ideation with plan:       False
  → Derived highest level:     3/5
  DR: BIS-10 impulsivity (newly extracted)
  BIS-10 total: 57.0 (normal)
  DR: Treatment + episode counts
  Lithium level:  None
  Valproate level: None
  ECT history:    True
  Lifetime episodes: {'depressive': None, 'manic': None, 'hypomanic': None, 'mixed': None}


## 5. Autism Spectrum Disorder (ASP / TSASDI)

ASP is structurally very different from the mood/psychosis cohorts. Key features:

- **Autism diagnostic profile**: DSM-5 type + 11 individual criteria (dsmaut01-11),
  DSM-5 domain flags, ADI-R, ADOS exam result
- **Developmental history**: motor milestones age, first phrases age, birth
  data, Apgar, perinatal flags (fetal distress, neonatal complications, etc.),
  parental ages at birth
- **Cognitive profile**: WAIS-IV total IQ + 4 index scores
- **Autism-specific instruments**: RBS-R (repetitive behaviors, 6 subscales),
  ADI-R (4 domain subscores), AQ-24, BRIEF (executive function, 9 subscales)
- **MCDD**: 15 Multiple Complex Developmental Disorder criteria
- **Learning disabilities**: dyslexia, dysorthography, dyscalculia, dysphasia,
  dyspraxia
- **BDI-II item 9** as partial suicide risk mitigation (no C-SSRS in ASP)

In [7]:
df_asp = load_cohort("ASP", nrows=1)
profile_asp, data_asp = build_vignette("ASP", df_asp.iloc[0])

print_header("ASP: Autism diagnosis (DSM-5 + ADI-R + ADOS)")
ad = data_asp.autism_diagnosis
print(f"  DSM type:        {ad.dsm_type_label}  (raw={ad.dsm_type})")
print(f"  DSM-5 domain 1:  {ad.dsm_domain1_met}  (social communication)")
print(f"  DSM-5 domain 2:  {ad.dsm_domain2_met}  (restricted behaviors)")
print(f"  ADI-R:           {ad.adi_diagnostic}")
print(f"  ADOS exam:       {ad.ados_exam}")
if ad.dsm_criteria:
    print("  DSM-5 individual criteria met:")
    for label, val in ad.dsm_criteria.items():
        if val:
            print(f"    ✓ {label}")

print_header("ASP: Developmental history")
dh = data_asp.developmental_history
print(f"  Motor milestones age:  {dh.age_motor_milestones} months")
print(f"  First phrases age:     {dh.age_first_phrases} months")
print(f"  Mother age at birth:   {dh.mother_age}")
print(f"  Father age at birth:   {dh.father_age}")
print(f"  Birth weight (g):      {dh.birth_weight_g}")
print(f"  Birth height (cm):     {dh.birth_height_cm}")
print(f"  Apgar:                 {dh.apgar_score}")
perinatal_flags = {
    "psychomotor delay": dh.psychomotor_delay,
    "language delay": dh.language_delay,
    "fetal distress": dh.fetal_distress,
    "neonatal complications": dh.neonatal_complications,
    "resuscitation": dh.resuscitation,
    "seizures": dh.seizures,
}
flagged = [k for k, v in perinatal_flags.items() if v]
print(f"  Perinatal flags:       {flagged or 'none'}")

print_header("ASP: Autism-specific instruments")
for domain, label in (
    ("cognitive_scores", "Cognitive (WAIS-IV)"),
    ("repetitive_behavior_scores", "Repetitive behaviors (RBS-R)"),
    ("autism_screening_scores", "Autism screening/diagnostic"),
    ("executive_function_scores", "Executive function (BRIEF)"),
    ("adhd_scores", "ADHD (ADHD-RS)"),
    ("anxiety_scores", "Anxiety (HAM-A, LSAS)"),
    ("depression_scores", "Depression (BDI-II)"),
):
    print(f"\n  {label}")
    for name, interp in getattr(data_asp, domain).items():
        if interp.score_available:
            print(f"    {name:12s}  {interp.raw_score!s:6s}  [{interp.severity_code}]")

print_header("ASP: MCDD + learning disabilities")
mcdd = data_asp.mcdd_profile
print(f"  MCDD criteria met: {mcdd.total_criteria_met}/{mcdd.total_criteria_assessed}")
ld = data_asp.learning_disabilities
ld_positive = [name for name in ("dyslexia", "dysorthography", "dyscalculia",
                                   "dysphasia", "dyspraxia", "speech_disorder",
                                   "stuttering") if getattr(ld, name)]
print(f"  Learning disabilities: {ld_positive or 'none'}")

print_header("ASP: Partial suicide risk (BDI-II item 9)")
print(f"  BDI-II item 9 (0-3): {data_asp.bdi_item9_suicidal_thoughts}")
print("  (ASP has no C-SSRS/ISF — BDI item 9 is the only suicide indicator)")

  ASP: Autism diagnosis (DSM-5 + ADI-R + ADOS)
  DSM type:        Syndrome d'Asperger  (raw=2.0)
  DSM-5 domain 1:  None  (social communication)
  DSM-5 domain 2:  None  (restricted behaviors)
  ADI-R:           1.0
  ADOS exam:       None
  ASP: Developmental history
  Motor milestones age:  13.0 months
  First phrases age:     60.0 months
  Mother age at birth:   25.0
  Father age at birth:   26.0
  Birth weight (g):      3540.0
  Birth height (cm):     54.0
  Apgar:                 10.0
  Perinatal flags:       ['language delay']
  ASP: Autism-specific instruments

  Cognitive (WAIS-IV)

  Repetitive behaviors (RBS-R)
    RBS-R         37.0    [moderate]

  Autism screening/diagnostic
    ADI-R         14.0    [unclassified]

  Executive function (BRIEF)

  ADHD (ADHD-RS)
    ADHD-RS       33.0    [moderate]

  Anxiety (HAM-A, LSAS)
    HAM-A         3.0     [normal]

  Depression (BDI-II)
  ASP: MCDD + learning disabilities
  MCDD criteria met: 3/15
  Learning disabilities: none
  

## 6. Inspecting the loaded YAML glossary

Everything the extractors read is declared in `config/glossary/` and loaded
lazily by `glossary_loader`. You can query the loaded config directly.

In [8]:
from face_rlvr.profiles.glossary_loader import (
    get_cohort_instruments,
    get_cohort_lab_ranges,
    get_cohort_column_map,
    get_clinical_constants,
)

COHORT = "BP"  # Change to SZ / DR / ASP to explore other cohorts

# 1. Instruments in the cohort's registry (common + cohort-specific merged)
instruments = get_cohort_instruments(COHORT.lower())
print_header(f"{COHORT}: {len(instruments)} instruments loaded from YAML")
for key, inst in list(instruments.items())[:10]:
    print(f"  {key:12s}  col={inst.total_column:22s}  {inst.domain}")
print(f"  ... and {len(instruments) - 10} more")

# 2. Severity bands for one specific instrument (from YAML)
example = instruments.get("MADRS") or next(iter(instruments.values()))
print_header(f"{example.name}: {example.full_name_fr}")
print(f"Column:     {example.total_column}")
print(f"Range:      {example.score_range}")
print("Severity bands (from YAML):")
for band in example.severity_thresholds:
    print(f"  {band.min_score:5.1f} – {band.max_score:5.1f}  [{band.code:15s}]  {band.label_fr}")

  BP: 29 instruments loaded from YAML
  MADRS         col=madrs_                  depression
  YMRS          col=ymrs_                   mania
  CGI-S         col=cgi01                   global_severity
  ASRM          col=altman_                 mania_self_report
  MAThyS        col=mathys_                 mood_self_report
  QIDS-SR16     col=qidsr120                depression_self_report
  FAST          col=fast_                   functioning
  EQ-5D         col=eq5d_                   quality_of_life
  STAI-YA       col=staya                   anxiety
  BIS-10        col=bis10_score_total       impulsivity
  ... and 19 more
  MADRS: Échelle de dépression de Montgomery-Åsberg
Column:     madrs_
Range:      (0.0, 60.0)
Severity bands (from YAML):
    0.0 –   6.0  [normal         ]  Pas de dépression / rémission
    7.0 –  19.0  [mild           ]  Dépression légère
   20.0 –  34.0  [moderate       ]  Dépression modérée
   35.0 –  60.0  [severe         ]  Dépression sévère


In [9]:
# 3. Lab ranges for the chosen cohort
labs = get_cohort_lab_ranges(COHORT.lower())
print_header(f"{COHORT}: {len(labs)} lab definitions")
for lab in labs[:12]:
    sex_flag = "  (sex-specific)" if lab.sex_specific else ""
    print(f"  {lab.csv_col:24s}  {lab.name_fr:30s}  {lab.normal_range} {lab.unit}{sex_flag}")

# 4. Column map — what CSV column holds which semantic field
cm = get_cohort_column_map(COHORT.lower())
print_header(f"{COHORT}: column_map.yaml (excerpt)")
print(f"  patient_id_column:              {cm.patient_id_column}")
print(f"  demographics.age:               {cm.demographics.age}")
print(f"  demographics.sex:               {cm.demographics.sex}")
print(f"  demographics.marital_status:    {cm.demographics.marital_status}")
if cm.vitals:
    print(f"  vitals.bmi:                     {cm.vitals.bmi}")
if cm.substance_use:
    print(f"  substance_use.tobacco:          {cm.substance_use.tobacco}")
if cm.family_history:
    print(f"  family_history.maternal:        {cm.family_history.maternal_psychiatric}")
    print(f"  family_history.relatives:       {len(cm.family_history.relatives)} prefixes")

  BP: 25 lab definitions
  gluc_lbstresc             Glycémie à jeun                 (3.9, 5.5) mmol/L
  hba1c_lbstresc            Hémoglobine glyquée             (4.0, 5.6) %
  chol_lbstresc             Cholestérol total               (0.0, 5.2) mmol/L
  hdl_lbstresc              HDL-cholestérol                 (1.0, 99.0) mmol/L
  ldl_lbstresc              LDL-cholestérol                 (0.0, 3.4) mmol/L
  trig_lbstresc             Triglycérides                   (0.0, 1.7) mmol/L
  tsh_lbstresc              TSH                             (0.4, 4.0) mUI/L
  t3fr_lbstresc             T3 libre                        (3.5, 6.5) pmol/L
  t4fr_lbstresc             T4 libre                        (12.0, 22.0) pmol/L
  creat_lbstresc            Créatinine                      (60.0, 110.0) µmol/L  (sex-specific)
  creatclr_lbstresc         Clairance créatinine            (90.0, 999.0) mL/min
  prolctn_lbstresc          Prolactine                      (2.0, 25.0) ng/mL
  BP: column_map.yam

In [10]:
# 5. Clinical constants (BMI, metabolic syndrome, Framingham, drug interactions)
cc = get_clinical_constants()

print_header("BMI categories (from YAML)")
for band in cc.bmi_categories:
    upper = f"<{band.max}" if band.max is not None else "any"
    print(f"  {upper:6s}  {band.label_fr}")

print_header("Drug-drug interactions (from YAML)")
for rule in cc.drug_interactions:
    print(f"  [{rule.severity:20s}]  {rule.drug1} + {rule.drug2}")

print_header("Medication-lab alert rules")
for rule in cc.medication_lab_alerts:
    print(f"  {rule.id:30s}  {rule.lab_names}")

print_header("Metabolic syndrome (IDF/ATP-III)")
ms = cc.metabolic_syndrome
print(f"  Min criteria for diagnosis: {ms.minimum_criteria_met}/5")
print(f"  Waist (M/F):                {ms.abdominal_obesity_cm.M} / {ms.abdominal_obesity_cm.F}")
print(f"  Triglycerides threshold:    {ms.hypertriglyceridemia_mmol_l}")
print(f"  HDL low (M/F):              {ms.hdl_low_mmol_l.M} / {ms.hdl_low_mmol_l.F}")
print(f"  Hypertension:               SBP ≥ {ms.hypertension.sbp_mmhg}, DBP ≥ {ms.hypertension.dbp_mmhg}")
print(f"  Hyperglycemia threshold:    {ms.hyperglycemia_mmol_l}")

  BMI categories (from YAML)
  <18.5   Insuffisance pondérale
  <25.0   Poids normal
  <30.0   Surpoids
  <35.0   Obésité classe I
  <40.0   Obésité classe II
  any     Obésité classe III (morbide)
  Drug-drug interactions (from YAML)
  [major               ]  on_lithium + nsaid
  [major               ]  on_lithium + on_carbamazepine
  [contre_indication   ]  on_clozapine + on_carbamazepine
  [moderate            ]  on_lithium + ieca
  [moderate            ]  on_valproate + on_lamotrigine
  Medication-lab alert rules
  lithium_creatinine_high         ['Créatinine', 'Creatinine']
  lithium_tsh_high                ['TSH']
  lithium_calcium_high            ['Calcium', 'Calcémie']
  valproate_liver_high            ['ALT', 'AST', 'GGT', 'ALAT', 'ASAT']
  valproate_platelets_low         ['Plaquettes', 'Platelets']
  clozapine_wbc_critical          ['Leucocytes', 'WBC']
  clozapine_wbc_low               ['Leucocytes', 'WBC']
  antipsychotic_prolactin_high    ['Prolactine', 'Prolactin']
  Meta

## 7. Multi-patient comparison across cohorts

Build a pandas DataFrame with headline indicators across multiple patients from
all 4 cohorts. Useful for cohort-level sanity checks and batch analyses.

In [11]:
N_PER_COHORT = 5
rows = []

for cohort in ("BP", "SZ", "DR", "ASP"):
    df = load_cohort(cohort, nrows=N_PER_COHORT)
    for _, csv_row in df.iterrows():
        profile, data = build_vignette(cohort, csv_row)

        # Common fields
        entry = {
            "cohort": cohort,
            "patient_id": data.patient_id,
            "age": data.demographics.age,
            "sex": data.demographics.sex,
            "n_abnormal_labs": sum(1 for lv in data.biology.values if lv.is_abnormal),
            "vignette_len": len(profile.full_vignette),
        }

        # Cohort-specific headline score
        if cohort == "BP":
            madrs = data.mood_scores.get("MADRS")
            ymrs = data.mood_scores.get("YMRS")
            entry["headline"] = f"MADRS={madrs.raw_score if madrs and madrs.score_available else '-'}  YMRS={ymrs.raw_score if ymrs and ymrs.score_available else '-'}"
        elif cohort == "SZ":
            panss = data.psychosis_scores.get("PANSS")
            entry["headline"] = f"PANSS={panss.raw_score if panss and panss.score_available else '-'}"
        elif cohort == "DR":
            tr = data.treatment_resistance
            cssrs = data.cssrs_assessment
            entry["headline"] = f"resist={tr.resistance_level or '-'}  cssrs={cssrs.highest_ideation_level or 0}/5"
        else:  # ASP
            ad = data.autism_diagnosis
            entry["headline"] = f"type={ad.dsm_type_label or '-'}"

        rows.append(entry)

pd.DataFrame(rows)

,cohort,patient_id,age,sex,n_abnormal_labs,vignette_len,headline
0,BP,10001,22,F,0,4754,MADRS=2.0 YMRS=2.0
1,BP,10002,30,F,0,4983,MADRS=7.0 YMRS=2.0
2,BP,10003,27,M,0,4639,MADRS=0.0 YMRS=0.0
3,BP,10004,32,M,1,4449,MADRS=0.0 YMRS=0.0
4,BP,10005,24,F,4,4839,MADRS=3.0 YMRS=1.0
5,SZ,10001,38,F,2,2694,PANSS=45.0
6,SZ,10002,63,F,6,4008,PANSS=67.0
7,SZ,10003,43,M,3,3025,PANSS=51.0
8,SZ,10004,20,F,1,3576,PANSS=90.0
9,SZ,10005,47,F,4,3695,PANSS=39.0


## 8. Live YAML mutation demo

Prove that the pipeline is truly YAML-driven: edit `config/glossary/bp/column_map.yaml`
in a subprocess, run extraction in a fresh Python process, and verify the
extracted value changes — all without touching any `.py` file.

(Uses a subprocess because Python caches module-level state; a fresh process
re-reads the YAML from disk.)

In [ ]:
import subprocess
import yaml

yaml_path = REPO_ROOT / "config" / "glossary" / "bp" / "column_map.yaml"
backup = yaml_path.read_text()

probe = '''
import pandas as pd
from face_rlvr.profiles import extract_bp_patient
df = pd.read_csv("data/BP.csv", nrows=1)
d = extract_bp_patient(df.iloc[0])
print(repr(d.demographics.age))
'''

# Baseline: read the current YAML
r1 = subprocess.run(["python", "-c", probe], capture_output=True, text=True, cwd=REPO_ROOT)
print(f"1. Baseline age:                    {r1.stdout.strip()}")

# Mutate: point demographics.age to a non-existent column
try:
    data = yaml.safe_load(backup)
    data["demographics"]["age"] = "nonexistent_column_xyz"
    yaml_path.write_text(yaml.dump(data, allow_unicode=True))

    r2 = subprocess.run(["python", "-c", probe], capture_output=True, text=True, cwd=REPO_ROOT)
    print(f"2. After YAML mutation:             {r2.stdout.strip()}")

    assert "None" in r2.stdout, "YAML mutation should make the age resolve to None"
    print("   ✓ YAML is the source of truth for column resolution")
finally:
    yaml_path.write_text(backup)

r3 = subprocess.run(["python", "-c", probe], capture_output=True, text=True, cwd=REPO_ROOT)
print(f"3. After YAML restore:              {r3.stdout.strip()}")

## 9. Full vignette dump for a single patient

Render the complete French vignette with all sections for any patient. The
vignette concatenates ~20 pre-rendered sections (synthesis, demographics,
history, mood, biology, treatment, comorbidities, trauma, ...).

In [12]:
COHORT = "BP"      # BP | SZ | DR | ASP
PATIENT_IDX = 0    # 0-based row index within the CSV

df = load_cohort(COHORT, nrows=PATIENT_IDX + 1)
profile, data = build_vignette(COHORT, df.iloc[PATIENT_IDX])

print_header(f"{COHORT}  patient_id={data.patient_id}  ({len(profile.full_vignette)} chars)")
print(profile.full_vignette)

  BP  patient_id=10001  (4754 chars)
Synthèse clinique : Patient suivie pour Bipolaire de type 2, actuellement euthymique (MADRS = 2). Facteurs notables : impulsivité élevée, troubles du sommeil.

Patient 10001, femme de 22 ans, suivie pour Bipolaire de type 2.
Célibataire, niveau d'études : bac.

Antécédents psychiatriques :
  - Début du trouble à 16 ans
  - Épisodes (vie entière) : 0 épisode(s) maniaque(s)
  - Cycles rapides : oui

Épisode actuel (critères DSM) :
  Critères dépressifs : 1/9 présents
    (modification pondérale)
  Critères maniaques : 0/9 présents
  Épisode le plus récent : Ne sais pas

Notes cliniques :
  * Discordance MADRS (2, pas de dépression / rémission) / QIDS-SR16 (8, dépression légère) : la auto-évaluation (QIDS) est plus sévère.
  * Discordance ASRM (7, positif) / YMRS (2, normal) : le patient rapporte des symptômes hypomaniaques non confirmés par le clinicien.

État thymique actuel :
  Hétéro-évaluation :
    - MADRS = 2 (Pas de dépression / rémission). Le 

## 10. Export a vignette to markdown for clinical review

In [16]:
COHORT = "ASP"
PATIENT_IDX = 0

df = load_cohort(COHORT, nrows=PATIENT_IDX + 1)
profile, data = build_vignette(COHORT, df.iloc[PATIENT_IDX])

out_path = REPO_ROOT / "output" / f"vignette_{COHORT}_{data.patient_id}.md"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(
    f"# Vignette — {COHORT} — {data.patient_id}\n\n{profile.full_vignette}\n",
    encoding="utf-8",
)
print(f"Written: {out_path}")

Written: /Users/andriikulakovskyi/Desktop/llm-rl/psych-dataset/output/vignette_ASP_10004.md
